## Método de Región de Confianza aplicado a ALMOST

### Aplicación a ALMOST

- **Suavidad:** $f\in C^\infty(\mathbb{R}^n)$.
- **Hessiana analítica:** $\nabla^2 f(x)$ se calcula mediante las expresiones del análisis anterior; en el óptimo $x^*=(1,\dots,1)$ es definida positiva (para $n=4$: autovalores $\approx 0.0871,\,2,\,2,\,45.913$).
- **Ventaja:** ALMOST tiene múltiples mínimos locales para $n\ge 2$ y su Hessiana puede volverse indefinida lejos de $x^*$. La región de confianza maneja esta situación sin modificar la Hessiana; el radio $\Delta_k$ actúa como mecanismo de globalización.
- **Subproblema:** se implementa el paso de Cauchy (suficiente para ilustrar el comportamiento). Para convergencia superlineal se usaría Dogleg o GC-Steihaug.

In [1]:
# Imports
import sys
from pathlib import Path

# Sube un nivel desde la carpeta del notebook hasta la raíz del repositorio
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
from funciones import almost_f, almost_grad, almost_hess, trust_region_cauchy

### Ejemplo de una iteración ($n=4$, $x_0=(0.5,0.5,0.5,0.5)^t$)

- $f(x_0)\approx 19.6289$, $\nabla f(x_0)\approx(-20.2344,-20.2344,-20.2344,-15.2344)^t$, $\|\nabla f(x_0)\|\approx 38.2153$.
- $\nabla^2 f(x_0)$ es definida positiva (todos sus autovalores $>0$).
- Con $\Delta_0=1$, el paso de Cauchy es $p_0^C\approx-\Delta_0\,g_0/\|g_0\|\approx(0.529,0.529,0.529,0.399)^t$.
- Se calcula $\rho_0$ y se ajusta $\Delta_1$.

### ¿Encuentra solución el método de Región de Confianza?

**Sí, con convergencia global.** A diferencia de SINGX, en ALMOST la Hessiana en el óptimo es **definida positiva**, por lo que el método converge sin dificultades. Con el **paso de Cauchy** la convergencia es **lineal** (el radio $\Delta_k$ se adapta bien y el método es globalmente robusto incluso si la Hessiana se vuelve indefinida lejos de $x^*$).

**Valor del mínimo encontrado:** el método alcanza el mínimo global
$$x^*=(1,1,1,1)^t,\qquad f(x^*)=0.$$
Con el paso de Cauchy se obtiene $f(x_k)\approx 10^{-10}$–$10^{-12}$ en $x_k\approx(1,1,1,1)$ (criterio de parada $\|\nabla f(x_k)\|<10^{-10}$). **El método no falla** para este punto inicial: la Hessiana se mantiene definida positiva a lo largo de la trayectoria y el subproblema tiene solución única en cada iteración.

In [2]:
# ----- Ejecución: Región de Confianza sobre ALMOST, n=4, x0=(0.5,0.5,0.5,0.5) -----
x0 = np.array([0.5, 0.5, 0.5, 0.5])

print(f"f(x0)       = {almost_f(x0):.8f}")       # esperado: 19.62890625
print(f"||grad(x0)||= {np.linalg.norm(almost_grad(x0)):.6f}")  # esperado: ≈ 38.215300
print(f"autovalores H(x0) = {np.linalg.eigvalsh(almost_hess(x0))}")  # todos > 0
print()

x_opt, hist = trust_region_cauchy(almost_f, almost_grad, almost_hess, x0,
                                   delta0=1.0, tol=1e-10, max_iter=500,
                                   verbose=True)

print(f"\n--- Resultado final ---")
print(f"x_opt       = {x_opt}")                # esperado: → (1,1,1,1)
print(f"f(x_opt)    = {almost_f(x_opt):.3e}")  # esperado: → 0
print(f"||grad||    = {np.linalg.norm(almost_grad(x_opt)):.3e}")
print(f"Iteraciones = {len(hist['f'])-1}")     # esperado: 10-30

f(x0)       = 19.62890625
||grad(x0)||= 38.214866
autovalores H(x0) = [ 0.44462812  2.46875     2.46875    36.74287188]

k=  0  f=1.207021e-03  ||g||=1.754e-01  Δ=2.000e+00  ρ=9.891e-01
k=  1  f=8.647365e-04  ||g||=1.852e-02  Δ=2.000e+00  ρ=1.001e+00
k=  2  f=5.862154e-04  ||g||=1.193e-01  Δ=2.000e+00  ρ=1.040e+00
k=  3  f=4.285194e-04  ||g||=1.187e-02  Δ=2.000e+00  ρ=1.001e+00
k=  4  f=3.009855e-04  ||g||=8.420e-02  Δ=2.000e+00  ρ=1.030e+00
k= 20  f=2.701211e-06  ||g||=7.765e-03  Δ=2.000e+00  ρ=1.004e+00
k= 40  f=9.831263e-09  ||g||=4.680e-04  Δ=2.000e+00  ρ=1.000e+00
k= 60  f=3.650043e-11  ||g||=2.851e-05  Δ=2.000e+00  ρ=1.000e+00
k= 80  f=1.356734e-13  ||g||=1.738e-06  Δ=2.000e+00  ρ=1.000e+00
k=100  f=5.043328e-16  ||g||=1.060e-07  Δ=2.000e+00  ρ=1.000e+00
k=120  f=1.874313e-18  ||g||=6.463e-09  Δ=2.000e+00  ρ=1.000e+00
Convergencia en 134 iteraciones, ||g||=9.27e-11

--- Resultado final ---
x_opt       = [1. 1. 1. 1.]
f(x_opt)    = 4.906e-20
||grad||    = 9.273e-11
Iteraciones = 1